# 🌍 Exploring the World Through Data

## Lecture Overview

In this lecture, we shift from working with small, pre-structured datasets to **real-world global data**.

We will:

- explore multiple datasets about the world  
- build **reusable analysis functions**  
- learn how to **validate and trust data**  
- combine datasets to create new insights  
- visualize data both over time and **on a world map**  

---

## Key Idea

> Real-world data analysis is not about one dataset —  
> it is about combining multiple sources and extracting meaning.

---

## What makes this different from previous lectures?

So far, we have:

- created visualizations  
- explored single datasets  
- worked with clean, structured data  

Now we will:

- work with **multiple datasets**
- handle **data quality issues**
- build **reusable tools**
- connect data to **geography**

---

## Learning Objectives

By the end of this session, you should be able to:

- inspect and validate real-world datasets  
- create reusable functions for analysis and visualization  
- visualize global data on a **world map**  
- merge datasets using common keys  
- compute new variables (e.g. GDP per capita)  
- interpret and communicate insights  

---

# 🌐 What is Our World in Data (OWID)?

**Our World in Data (OWID)** is an open platform that provides:

- global datasets on topics such as:
  - population  
  - health  
  - economy  
  - environment  
- interactive visualizations  
- downloadable datasets for analysis  

-> [Our World in Data](https://ourworldindata.org/)

---

## Why OWID is useful

OWID is a powerful resource because:

- data is **well-structured and documented**
- datasets are **consistent across countries and years**
- it allows both:
  - quick exploration (via website)
  - deeper analysis (via downloaded data)

---

## Two ways to use OWID

### 1. Explore directly on the website

You can:

- interact with charts  
- filter countries  
- change variables  
- explore trends quickly  

-> This is useful for **idea generation**

---

### 2. Download and analyze the data

You can:

- download datasets as `.csv`
- load them into Python
- perform your own analysis
- create custom visualizations

-> This is what we will do in this lecture

---

## Key takeaway

> OWID allows us to move from **exploring data visually**  
> to **analyzing and building insights ourselves**

---

# Important Mindset

As we work through this lecture, always ask:

- What does this data represent?
- Can I trust this data?
- What might be missing or incorrect?
- What question am I trying to answer?

---


# Questions we will explore

- How has population changed over time?
- Which countries are the richest?
- Do richer countries live longer?
- How do global patterns differ across regions?

---

You can start by installing some additional libraries we will use today
- `geopandas` for working with geographic data
  - `folium` for creating interactive maps
  - `mapclassify` for classifying data for mapping

Run the following command, after which you can delete the cell:
```python
!pip install geopandas "folium>=0.12" mapclassify pycountry_convert
```

# 1. Population Dataset

## Goal

We begin our analysis with the **population dataset**.

This will allow us to:

- understand the structure of real-world data  
- practice data inspection and validation  
- begin building reusable analysis steps  

---

## Important Practice

Before doing any analysis, we want to:

- give meaningful names to our data  
- understand what each column represents  

> Clear naming makes analysis easier, especially when working with multiple datasets.

---

# Load the Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

population_df = pd.read_csv("./data/population.csv")
population_df.head()

### First Inspection

Before doing any analysis, we inspect the dataset.

In [ ]:
population_df.info()

In [ ]:
population_df.describe()

### Initial Questions

- What is the time range of the dataset?
- How many countries are included per year?
- Are there missing values?
- Do the values look reasonable?

## Data Inspection

### Missing Values

First, we check for missing values in the dataset.

#### Interpretation

- Are there missing values?
- If yes, how many?
- In which columns?

#### Possible strategies

- remove missing values  
- fill them (if appropriate)  
- leave them but be aware  

In [ ]:
population_df.isna().sum()

In [ ]:
population_df[population_df.isna().any(axis=1)]['country'].unique()

Very interesting! OWID provides aggregate data as well, where it combines countries into regions (e.g. "Americas (UN)", "Less developed regions", etc.). These aggregate rows have missing values for the `country_code` column, which is expected since they do not represent individual countries.

```
Tip: OWID also provides continent level aggregates, which can be useful for regional analysis.
```

### Minimum and Maximum Values

Do the values in the `population` column look reasonable?

In [ ]:
population_df["population"].min(), population_df["population"].max()

### Time Range

What is the time range of the dataset?

In [ ]:
population_df["year"].min(), population_df["year"].max()

## Question

How has the **total world population** changed over time?

In [ ]:
global_population = (
    population_df
    .groupby("year")["population"]
    .sum()
    .reset_index()
)

global_population.head()

In [ ]:
from functions import remove_aggregate_rows

population_df = remove_aggregate_rows(population_df)
global_population = (
    population_df
    .groupby("year")["population"]
    .sum()
    .reset_index()
)

global_population.tail()

### Task

- visualize the total world population over time using a line plot

In [ ]:
# Your code goes here

###  Improve readability

Population numbers are very large, which makes them hard to interpret.

We can convert them to **billions** for better readability.

In [ ]:
# Your code goes here

- Does the growth appear linear or exponential?
- Are there periods of faster or slower growth?

In [ ]:
global_population["growth_rate"] = (
    global_population["population"]
    .pct_change()
    * 100
)

global_population.head()

### Task
- visualize the growth rate of the world population over time using a line plot
- use seaborn's `lineplot`

In [ ]:
# Your code goes here

# Discussion

- Is the population growth rate increasing or decreasing?
- When was growth the fastest?
- Is global population growth slowing down?
- What might explain changes over time?

---

# Key Insight

> Even if total population keeps increasing,  
> the **growth rate can decrease over time**.

This is an important distinction between:

- absolute growth  
- relative growth  

In [ ]:
from functions import add_continent_column

population_df = add_continent_column(population_df)
population_df.head()

In [ ]:
continent_population = (
    population_df
    .groupby(["year", "continent"], as_index=False)["population"]
    .sum()
)

continent_population['population_million'] = continent_population['population'] / 1e6
continent_population.head()

### Task
- visualize the world population by continent over time using a line plot
- use seaborn's `lineplot`

In [ ]:
# Your code goes here

In [ ]:
continent_population = continent_population.sort_values(["continent", "year"])

continent_population["growth_rate"] = (
    continent_population
    .groupby("continent")["population_million"]
    .pct_change()
    * 100
)

continent_population.head()

### Task
- create a line plot of continental population growth over time
- use seaborn's `lineplot`

In [ ]:
# Your code goes here

# Discussion

- Which continents have the largest populations?
- Which continents are growing the fastest?
- Is growth slowing down globally or only in certain regions?

# 2. Geographic Visualization with GeoPandas

## Goal

So far, we have looked at:

- global trends  
- continent-level trends  

Now we introduce a new perspective:

> Where is the population located?

To answer this, we visualize data on a **world map**.

---

# What is GeoPandas?

GeoPandas extends pandas to work with **geographic data**.

It allows us to:

- load map data (countries, regions)
- merge it with our dataset
- visualize values on a map

---

# Step 1 — Load World Map Data

We load a dataset containing country geometries.

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(
    "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_110m_admin_0_countries.geojson"
)

gdf.head()

In [ ]:
gdf.columns

We start with a simple map.

In [ ]:
gdf.plot(figsize=(10, 6))
plt.title("World Map")
plt.show()

## Merge Population Data with Map

We need to:

1. select a specific year  
2. merge population data with map data  

---

## Select a year

In [ ]:
year = 2020

population_year = population_df[population_df["year"] == year]

## Merge datasets

In the `gdf` GeoDataFrame, we have both the geometry of each country and its population for the selected year. We merge based on the `adm0_a3` country code, which is a common identifier for countries in geographic datasets, which also exists in the OWID population dataset as `country_code`. This allows us to combine the population data with the geographic data for each country.

In [ ]:
merged = gdf[['admin', 'adm0_a3', 'geometry']].merge(
    population_year,
    left_on="adm0_a3",
    right_on="country_code",
    how="left"
)

merged.head()

In [ ]:
import numpy as np

merged.plot(
    column="population",
    cmap="viridis",
    legend=True,
    figsize=(12, 6),
    missing_kwds={"color": "lightgrey"}
)

plt.title(f"Log Population by Country ({year})")
plt.show()

### Improve Readability

Population values vary widely, making visualization difficult.

We can use a logarithmic transformation.

In [ ]:
import numpy as np

merged["log_population"] = np.log10(merged["population"])

merged.plot(
    column="log_population",
    cmap="viridis",
    legend=True,
    figsize=(12, 6),
    missing_kwds={"color": "lightgrey"}
)

plt.title(f"Log Population by Country ({year})")
plt.show()

## Interactive Map with `.explore()`

GeoPandas also supports interactive maps.


In [ ]:
merged.explore(
    column="log_population",
    cmap="viridis",
    missing_kwds={"color": "lightgrey"})

We will now:

- turn this process into a reusable function  
- apply it to other datasets (GDP, life expectancy)  